# Small Deep Learning Models in Google Colab with TensorFlow and Keras

This notebook introduces small deep learning models for tabular data using **TensorFlow** and the **Keras** interface in **Google Colab**. The main goal is to show how to build, train, and compare a **basic MLP** and a **deep MLP** for a binary classification problem using the **Bank Marketing** dataset from the UCI Machine Learning Repository, stored in a CSV file called `bank-full.csv`.

We will use:
- a mix of categorical and numerical predictors
- a binary target variable
- a training and validation split created inside the notebook

At the end, very small examples of **CNN** and **RNN** models are also provided so that students can see how other neural network architectures are built in Keras.


## Introduction: TensorFlow, PyTorch, and Keras

**TensorFlow** is a widely used open-source framework for machine learning and deep learning. It provides the tools needed to define, train, and deploy neural network models.

**PyTorch** is another major deep learning framework. It is especially popular in research and experimentation because of its flexibility and intuitive design.

**Keras** is a high-level neural network interface that runs within TensorFlow. It allows us to build deep learning models in a simpler and more readable way. In teaching and small Google Colab examples, Keras is often the easiest starting point because it reduces coding complexity while still allowing powerful models to be built.

In this notebook, we will use **TensorFlow with the Keras interface**.


## Using GPU in Google Colab

Google Colab allows notebooks to run on a **GPU**, which can make neural network training faster.

To enable a GPU in Colab:

1. Go to **Runtime**
2. Select **Change runtime type**
3. Under **Hardware accelerator**, choose **GPU**
4. Click **Save**

We can then check whether TensorFlow can detect the GPU.


In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPUs detected:", tf.config.list_physical_devices("GPU"))

## Common Keras layers used in neural networks

Before building models, it is useful to understand the role of some common Keras layers.

### 1. Dense layer
A **Dense** layer is the standard fully connected layer used in MLP networks. Each neuron in the layer receives input from all outputs of the previous layer.

Example:
```python
layers.Dense(16, activation="relu")
```

This means:
- 16 neurons in the layer
- `relu` as the activation function

### 2. Dropout layer
A **Dropout** layer randomly switches off a proportion of neurons during training. This helps reduce **overfitting** by preventing the model from depending too heavily on a small number of neurons.

Example:
```python
layers.Dropout(0.2)
```

This means 20% of neurons are temporarily dropped during training.

### 3. Input layer
The **Input** layer tells Keras the shape of the input data.

Example:
```python
layers.Input(shape=(input_dim,))
```

This means each observation has `input_dim` predictor values after preprocessing.

### 4. Output layer
For a **binary classification** problem, the output layer usually has:
- 1 neuron
- `sigmoid` activation

Example:
```python
layers.Dense(1, activation="sigmoid")
```

This produces a predicted probability between 0 and 1.

### 5. Activation functions
Activation functions allow neural networks to learn non-linear relationships.

Common examples:
- **ReLU**: often used in hidden layers
- **Sigmoid**: often used in binary output layers
- **Softmax**: often used in multi-class output layers


## Overview of the modelling task

In this notebook, we will:

1. Load a CSV file called `bank-full.csv`
2. Use the UCI Bank Marketing dataset
3. Predict the binary target `y`
4. Split the data into training and validation sets
5. Preprocess categorical and numerical predictors
6. Train:
   - a **basic MLP**
   - a **deep MLP**
7. Compare their performance using:
   - validation loss
   - validation accuracy
   - validation misclassification rate


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from tensorflow import keras
from tensorflow.keras import layers

## Load the dataset

Make sure the file `bank-full.csv` has been uploaded into the Colab session before running the next cell.

This is the UCI Bank Marketing dataset. In this dataset, the target variable `y` indicates whether the client subscribed to a term deposit.


In [ ]:
df = pd.read_csv("bank-full.csv", sep=";")
df.head()

In [ ]:
df.info()

## Define target and predictors

The target variable is `y`.

We will use the following **categorical predictors**:
- `job`
- `marital`
- `education`
- `default`
- `housing`
- `loan`
- `contact`
- `month`
- `poutcome`

We will use the following **numerical predictors**:
- `age`
- `balance`
- `day`
- `duration`
- `campaign`
- `pdays`
- `previous`

**Note:** In practice, the variable `duration` is often excluded from real campaign prediction models because it is only known after the contact has taken place. Here we keep it for teaching purposes.


In [ ]:
target = "y"

categorical_features = [
    "job", "marital", "education", "default",
    "housing", "loan", "contact", "month", "poutcome"
]

numerical_features = [
    "age", "balance", "day", "duration",
    "campaign", "pdays", "previous"
]

all_features = categorical_features + numerical_features

df = df[all_features + [target]].copy()
df.head()

## Convert the target variable

The target `y` contains values such as `yes` and `no`.  
We convert these into 1 and 0.


In [ ]:
df[target] = df[target].map({"yes": 1, "no": 0})
df[target].value_counts()

## Create training and validation datasets

Unlike the earlier teaching dataset, `bank-full.csv` does not contain a predefined partition column, so we will create our own split using `train_test_split`.

We will:
- use 70% of the data for training
- use 30% of the data for validation
- use `stratify=y` so that the class balance is preserved across both sets


In [ ]:
X_raw = df[all_features]
y = df[target]

X_train_raw, X_valid_raw, y_train, y_valid = train_test_split(
    X_raw, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train_raw.shape[0])
print("Validation rows:", X_valid_raw.shape[0])

## Pre-process the data

Neural networks require numerical input, so we must preprocess the predictors.

We will:
- one-hot encode the categorical variables
- standardise the numerical variables

This is important because:
- categorical values must be converted into numbers
- scaling numerical variables often improves neural network training


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numerical_features)
    ]
)

X_train = preprocessor.fit_transform(X_train_raw)
X_valid = preprocessor.transform(X_valid_raw)

# Convert to dense arrays for Keras
X_train = X_train.toarray() if hasattr(X_train, "toarray") else X_train
X_valid = X_valid.toarray() if hasattr(X_valid, "toarray") else X_valid

print("Training matrix shape:", X_train.shape)
print("Validation matrix shape:", X_valid.shape)

## Build a basic MLP model

A **Multi-Layer Perceptron (MLP)** is a feed-forward neural network made up of fully connected layers.

Our **basic MLP** will contain:
- one input layer
- one hidden dense layer
- one output layer

This is a useful starting benchmark before moving to a deeper network.


In [ ]:
input_dim = X_train.shape[1]

basic_mlp = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

basic_mlp.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

basic_mlp.summary()

## Train the basic MLP

We will use **early stopping** so that training stops if the validation loss does not improve for several epochs. This helps reduce overfitting.


In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history_basic = basic_mlp.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=50,
    batch_size=32,
    verbose=1,
    callbacks=[early_stop]
)

## Evaluate the basic MLP

We will calculate:
- validation accuracy
- validation misclassification rate

The **misclassification rate** is:

\[
Misclassification Rate = 1 - Accuracy
\]

It tells us the proportion of validation observations classified incorrectly.


In [ ]:
basic_probs = basic_mlp.predict(X_valid)
basic_pred = (basic_probs >= 0.5).astype(int).ravel()

basic_acc = accuracy_score(y_valid, basic_pred)
basic_misclass = 1 - basic_acc

print("Basic MLP validation accuracy:", round(basic_acc, 4))
print("Basic MLP validation misclassification rate:", round(basic_misclass, 4))

In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(y_valid, basic_pred))

print("\nClassification Report:")
print(classification_report(y_valid, basic_pred, digits=4))

## Build a deep MLP model

We now extend the basic MLP into a **deep MLP** by adding more hidden layers.

A deeper network can learn more complex relationships, although it may also increase the risk of overfitting. To help manage this, we add **Dropout** layers.

Our deep MLP will contain:
- one input layer
- three hidden dense layers
- two dropout layers
- one output layer


In [ ]:
deep_mlp = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

deep_mlp.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

deep_mlp.summary()

## Exercise

Modify the deep MLP in two ways:
1. Change the first hidden layer from `64` neurons to `32`
2. Change the dropout rate from `0.2` to `0.3`

Then retrain the model and compare the validation misclassification rate.


## Train the deep MLP

In [ ]:
history_deep = deep_mlp.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=50,
    batch_size=32,
    verbose=1,
    callbacks=[early_stop]
)

## Evaluate the deep MLP

In [ ]:
deep_probs = deep_mlp.predict(X_valid)
deep_pred = (deep_probs >= 0.5).astype(int).ravel()

deep_acc = accuracy_score(y_valid, deep_pred)
deep_misclass = 1 - deep_acc

print("Deep MLP validation accuracy:", round(deep_acc, 4))
print("Deep MLP validation misclassification rate:", round(deep_misclass, 4))

In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(y_valid, deep_pred))

print("\nClassification Report:")
print(classification_report(y_valid, deep_pred, digits=4))

## Compare the basic and deep MLP models

A lower validation misclassification rate indicates better validation performance.


In [ ]:
comparison = pd.DataFrame({
    "Model": ["Basic MLP", "Deep MLP"],
    "Validation Accuracy": [basic_acc, deep_acc],
    "Validation Misclassification Rate": [basic_misclass, deep_misclass]
})

comparison

## Plot training and validation history

These plots are useful for diagnosing:
- **underfitting**
- **overfitting**
- whether the deep model is learning more effectively than the basic model


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history_basic.history["loss"], label="Basic train loss")
plt.plot(history_basic.history["val_loss"], label="Basic val loss")
plt.plot(history_deep.history["loss"], label="Deep train loss")
plt.plot(history_deep.history["val_loss"], label="Deep val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history_basic.history["accuracy"], label="Basic train accuracy")
plt.plot(history_basic.history["val_accuracy"], label="Basic val accuracy")
plt.plot(history_deep.history["accuracy"], label="Deep train accuracy")
plt.plot(history_deep.history["val_accuracy"], label="Deep val accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.show()

## Exercise

Modify the comparison in two ways:
1. Change `epochs=50` to `epochs=100`
2. Change `batch_size=32` to `batch_size=64`

Then retrain and check whether the deep MLP improves.


## Brief interpretation

The basic MLP acts as a simple neural network benchmark. The deep MLP is more flexible because it contains additional hidden layers and dropout regularisation. However, on structured tabular data, deeper models do not always outperform simpler ones. This is why comparing validation performance is important. The validation misclassification rate gives a direct and easy-to-interpret summary of the proportion of incorrect predictions.


## Very basic CNN example

A **Convolutional Neural Network (CNN)** is mainly designed for image data. It is not the natural first choice for ordinary tabular data such as `bank-full.csv`, but it is useful to see a basic example on a public image dataset such as **MNIST**.

A CNN uses layers such as:
- `Conv2D` for feature detection
- `MaxPooling2D` for dimensionality reduction
- `Flatten` to convert image features into a vector before dense layers


In [ ]:
from tensorflow.keras.datasets import mnist

(X_train_img, y_train_img), (X_test_img, y_test_img) = mnist.load_data()

# Show a sample of images
plt.figure(figsize=(8, 4))
for i in range(6):
    plt.subplot(2, 3, i+1)
    plt.imshow(X_train_img[i], cmap="gray")
    plt.title(f"Label: {y_train_img[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

X_train_img = X_train_img.astype("float32") / 255.0
X_test_img = X_test_img.astype("float32") / 255.0

# Add channel dimension
X_train_img = np.expand_dims(X_train_img, -1)
X_test_img = np.expand_dims(X_test_img, -1)

cnn_model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(16, kernel_size=(3, 3), activation="relu"),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

cnn_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn_model.fit(
    X_train_img, y_train_img,
    validation_split=0.1,
    epochs=3,
    batch_size=64
)

## Exercise

Modify the CNN in two ways:
1. Change the first `Conv2D` layer from `16` filters to `32`
2. Change `epochs=3` to `epochs=5`

Then retrain and observe whether validation accuracy improves.


## Very basic RNN example

A **Recurrent Neural Network (RNN)** is designed for sequential data such as:
- text
- language
- time series

It is usually not the preferred model for tabular data, but it is useful to see a simple example on a public text dataset such as **IMDB movie reviews**.

An RNN example often uses layers such as:
- `Embedding` to convert word indices into dense vectors
- `SimpleRNN`, `LSTM`, or `GRU` to process sequences
- `Dense` for final prediction


In [ ]:
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_features = 10000
maxlen = 200

(X_train_seq, y_train_seq), (X_test_seq, y_test_seq) = imdb.load_data(num_words=max_features)

# Show a few raw samples
for i in range(3):
    print("Sequence:", X_train_seq[i][:20], "...")  # first 20 tokens
    print("Label:", y_train_seq[i])
    print("-"*40)

# Decode and show sample reviews
word_index = keras.datasets.imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

def decode_review(sequence):
    return " ".join([reverse_word_index.get(i - 3, "?") for i in sequence])

for i in range(3):
    print("Review:", decode_review(X_train_seq[i])[:300], "...")
    print("Label:", y_train_seq[i])
    print("-"*50)

# Pad sequences
X_train_seq = pad_sequences(X_train_seq, maxlen=maxlen)
X_test_seq = pad_sequences(X_test_seq, maxlen=maxlen)

rnn_model = keras.Sequential([
    layers.Input(shape=(maxlen,)),
    layers.Embedding(input_dim=max_features, output_dim=32),
    layers.SimpleRNN(16),
    layers.Dense(1, activation="sigmoid")
])

rnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

rnn_model.fit(
    X_train_seq, y_train_seq,
    validation_split=0.1,
    epochs=3,
    batch_size=64
)

## Exercise

Modify the RNN in two ways:
1. Replace `SimpleRNN(16)` with `LSTM(16)`
2. Change `epochs=3` to `epochs=5`

Then retrain and compare the validation accuracy.


## Final summary

In this notebook, we used **TensorFlow with the Keras interface** to build small deep learning models in Google Colab. We loaded the file `bank-full.csv`, used the UCI Bank Marketing dataset, created training and validation datasets, and trained both a **basic MLP** and a **deep MLP** for a binary classification target. We compared the models using validation accuracy and validation misclassification rate, and we introduced important Keras layers such as **Dense**, **Dropout**, **Input**, and output layers with **sigmoid** activation. Finally, we included very small **CNN** and **RNN** examples so that students could see how Keras supports different neural network architectures for different data types.


In [ ]:
# Save the model comparison table if needed
comparison.to_csv("mlp_model_comparison.csv", index=False)